# Neo4j + PyTorch Geometric GNN Benchmark

This Colab notebook provides a clean benchmark for **Basic GNN, GCN, GraphSAGE, GAT, and GIN** on the **Cora** node-classification dataset.

The notebook is designed as a practical starting point for a Neo4j-based graph ML use case:

```text
Neo4j Graph
    |
    v
Graph structure + node features
    |
    v
PyTorch Geometric
    |
    v
GNN model
    |
    v
Node classification
    |
    v
Evaluation
```

**Important:** Neo4j Community Edition is not automatically available inside Colab. The Neo4j section is therefore optional and requires a reachable Neo4j instance and credentials.

In [ ]:
# 1. Environment check and PyTorch Geometric installation
import sys
import subprocess
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# Current PyG releases can be installed directly in Colab.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch-geometric"])

import torch_geometric
print("PyTorch Geometric:", torch_geometric.__version__)

In [ ]:
# 2. Imports and reproducibility
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.datasets import Planetoid
from torch_geometric.nn import (
    MessagePassing,
    GCNConv,
    SAGEConv,
    GATConv,
    GINConv,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 3. Load the Cora graph

Cora is a citation network used for node classification.

- Nodes represent scientific papers.
- Edges represent citation relationships.
- Node features represent paper word information.
- The target is the paper class.

This is a benchmark dataset for validating the GNN implementation before moving to a network-security graph.

In [ ]:
dataset = Planetoid(root="/content/data/Cora", name="Cora")
data = dataset[0].to(device)

print("Dataset:", dataset.name)
print("Number of classes:", dataset.num_classes)
print("Number of node features:", dataset.num_node_features)
print("Nodes:", data.num_nodes)
print("Edges:", data.num_edges)
print("Train nodes:", int(data.train_mask.sum()))
print("Validation nodes:", int(data.val_mask.sum()))
print("Test nodes:", int(data.test_mask.sum()))

## 4. GNN architectures

The benchmark uses the same graph, features, train/validation/test masks, optimizer, and training budget for each model.

- **Basic GNN:** simple message passing baseline.
- **GCN:** graph convolution using normalized neighborhood aggregation.
- **GraphSAGE:** samples/aggregates neighborhood information.
- **GAT:** uses attention to weight neighboring nodes.
- **GIN:** uses an MLP-based aggregation designed for strong graph-structure discrimination.

In [ ]:
class BasicGNNLayer(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr="mean")
        self.lin = nn.Linear(in_channels, out_channels)

    def forward(self, x, edge_index):
        return self.lin(self.propagate(edge_index, x=x))

    def message(self, x_j):
        return x_j


class BasicGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = BasicGNNLayer(in_channels, hidden_channels)
        self.conv2 = BasicGNNLayer(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        return self.conv2(x, edge_index)


class GCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.5, training=self.training)
        return self.conv2(x, edge_index)


class GraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.5, training=self.training)
        return self.conv2(x, edge_index)


class GAT(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=2, concat=True, dropout=0.2)
        self.conv2 = GATConv(hidden_channels * 2, out_channels, heads=1, concat=False)

    def forward(self, x, edge_index):
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.5, training=self.training)
        return self.conv2(x, edge_index)


class GIN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()

        mlp1 = nn.Sequential(
            nn.Linear(in_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels),
        )
        mlp2 = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, out_channels),
        )

        self.conv1 = GINConv(mlp1)
        self.conv2 = GINConv(mlp2)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.5, training=self.training)
        return self.conv2(x, edge_index)


models = {
    "Basic GNN": BasicGNN,
    "GCN": GCN,
    "GraphSAGE": GraphSAGE,
    "GAT": GAT,
    "GIN": GIN,
}

In [ ]:
# 5. Training and evaluation functions
def evaluate(model, data, mask):
    model.eval()
    with torch.no_grad():
        logits = model(data.x, data.edge_index)
        pred = logits.argmax(dim=1)
        correct = (pred[mask] == data.y[mask]).sum().item()
        total = int(mask.sum())
    return correct / total


def train_model(model_class, data, epochs=150, lr=0.01, weight_decay=5e-4):
    torch.manual_seed(SEED)

    model = model_class(
        data.num_node_features,
        64,
        dataset.num_classes
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    best_val = 0.0
    best_state = None
    losses = []

    start = time.perf_counter()

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()

        logits = model(data.x, data.edge_index)
        loss = F.cross_entropy(logits[data.train_mask], data.y[data.train_mask])

        loss.backward()
        optimizer.step()

        val_acc = evaluate(model, data, data.val_mask)

        if val_acc > best_val:
            best_val = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        losses.append(float(loss.item()))

    elapsed = time.perf_counter() - start

    if best_state is not None:
        model.load_state_dict(best_state)

    test_acc = evaluate(model, data, data.test_mask)

    return model, {
        "test_accuracy": test_acc,
        "best_val_accuracy": best_val,
        "training_time_sec": elapsed,
        "parameters": sum(p.numel() for p in model.parameters()),
        "losses": losses,
    }

In [ ]:
# 6. Run the benchmark
results = {}
trained_models = {}

for name, model_class in models.items():
    print(f"Training {name}...")
    model, metrics = train_model(model_class, data)
    trained_models[name] = model
    results[name] = metrics
    print(
        f"  Test accuracy: {metrics['test_accuracy']:.4f} | "
        f"Best validation: {metrics['best_val_accuracy']:.4f} | "
        f"Time: {metrics['training_time_sec']:.2f}s"
    )

results_df = pd.DataFrame([
    {
        "Model": name,
        "Test Accuracy": metrics["test_accuracy"],
        "Best Validation Accuracy": metrics["best_val_accuracy"],
        "Training Time (sec)": metrics["training_time_sec"],
        "Parameters": metrics["parameters"],
    }
    for name, metrics in results.items()
]).sort_values("Test Accuracy", ascending=False)

results_df

In [ ]:
# 7. Compare results visually
plt.figure(figsize=(10, 5))
plt.bar(results_df["Model"], results_df["Test Accuracy"])
plt.ylabel("Test Accuracy")
plt.title("Cora Node Classification: GNN Architecture Comparison")
plt.xticks(rotation=20)
plt.ylim(0, 1)
plt.show()

In [ ]:
# 8. Plot training loss
plt.figure(figsize=(10, 5))

for name, metrics in results.items():
    plt.plot(metrics["losses"], label=name)

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss Comparison")
plt.legend()
plt.show()

## 9. Neo4j integration

For the practical Vehere direction, Neo4j can be the graph storage layer.

Example network-security graph:

```text
(:Host)-[:CONNECTS_TO]->(:Host)
(:Host)-[:CONNECTS_TO]->(:IP)
(:Host)-[:RESOLVES_TO]->(:Domain)
(:Host)-[:USES_PROTOCOL]->(:Protocol)
```

The GNN can then learn from graph structure and node features.

The following cell is optional. It does not require Neo4j for the Cora benchmark.

In [ ]:
# 10. Optional Neo4j connection
# Set these environment variables in Colab before running this cell:
#
# NEO4J_URI=neo4j+s://<host>
# NEO4J_USERNAME=neo4j
# NEO4J_PASSWORD=<password>
#
# For Neo4j Community Edition, the instance must be reachable from Colab.

import os

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

if NEO4J_URI and NEO4J_PASSWORD:
    try:
        from neo4j import GraphDatabase

        driver = GraphDatabase.driver(
            NEO4J_URI,
            auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
        )

        with driver.session() as session:
            record = session.run("RETURN 'Neo4j connection successful' AS message").single()
            print(record["message"])

        driver.close()

    except Exception as e:
        print("Neo4j connection failed:", repr(e))
else:
    print("Neo4j credentials not configured. Skipping connection test.")

In [ ]:
# 11. Example Neo4j network graph schema for the future use case

neo4j_schema = {
    "nodes": ["Host", "IP", "Domain", "Protocol"],
    "relationships": [
        "Host-[:CONNECTS_TO]->Host",
        "Host-[:CONNECTS_TO]->IP",
        "Host-[:RESOLVES_TO]->Domain",
        "Host-[:USES_PROTOCOL]->Protocol",
    ],
    "example_features": [
        "connection_count",
        "unique_destination_count",
        "unique_port_count",
        "bytes_sent",
        "bytes_received",
        "dns_request_count",
        "failed_connection_count",
        "external_connection_count",
    ],
}

neo4j_schema

## 12. Proposed Vehere proof of concept

The Cora benchmark is only the architecture validation step.

The next practical experiment can be:

```text
Network telemetry
       |
       v
Neo4j network graph
       |
       v
Node features + relationships
       |
       v
PyTorch Geometric
       |
       v
GraphSAGE / GCN / GAT / GIN
       |
       v
Suspicious Host Classification
       |
       v
Prediction stored in Neo4j
       |
       v
Graph traversal for investigation
       |
       v
GraphRAG / LLM explanation
```

### Potential technical value

- Detect graph-based patterns rather than isolated events.
- Use neighboring entities as model input.
- Rank suspicious hosts or other network entities.
- Combine GNN prediction with Neo4j graph investigation.
- Provide graph evidence to a future GraphRAG workflow.

### Potential business value

The intended value is to automate repetitive correlation and investigation steps, helping analysts reach relevant evidence faster. Any claimed reduction in investigation time should be validated with representative Vehere data.

## 13. Key benchmark conclusion

The benchmark provides a controlled comparison of five GNN approaches on the same graph classification task.

The next decision should not be based only on the highest Cora accuracy. For a Vehere use case, model selection should also consider:

- Precision and recall for suspicious activity.
- F1 score.
- Inference latency.
- Scalability to large graphs.
- Ability to handle new network entities.
- Integration complexity with Neo4j.
- Usefulness of the resulting prediction for investigation.

This keeps the research focused on a practical Neo4j + GNN security workflow rather than only a benchmark score.